# Qwen2.5-VL 多模态模型 Demo (.ipynb)

> 使用 HuggingFace Hub 下载一个轻量多模态模型（`Qwen/Qwen2.5-VL-3B-Instruct`），
> 实现"用户字符输入 → 模型字符输出"的完整流程，包含：
> 1. 字符输入 + 非流式输出
> 2. 字符输入 + 流式输出
> 3. 多模态输入（本地图片）+ 字符输出（图片描述）


In [13]:
import os 
os.environ["HF_HUB_OFFLINE"] = "1"  # 开启离线模式，强制只从本地缓存加载模型，禁止联网下载
os.environ["TRANSFORMERS_OFFLINE"] = "1"  # 开启Transformers离线模式，跳过版本检查直接使用本地文件

In [14]:
# ------------------------------------------------------------------------------
# 1. 导入标准库与第三方库
# ------------------------------------------------------------------------------

import torch  # PyTorch：张量计算与 GPU 加速；本脚本用其自动选择设备、数据类型
from PIL import Image  # Pillow：图像库；用于打开本地图片并转成模型可处理的格式
from transformers import (
    AutoProcessor,   # 自动根据模型名加载对应的 processor（含 tokenizer + 图像预处理）
    AutoModelForImageTextToText,  # 自动加载"视觉→文本"类模型（多模态模型专用类）
)
# from huggingface_hub import snapshot_download  # 官方下载工具：把整个模型仓库下载到本地
import pprint

In [15]:
# ------------------------------------------------------------------------------
# 2. 配置区：模型名称、本地缓存路径、生成参数
# ------------------------------------------------------------------------------
MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"  # 模型在 HuggingFace Hub 的唯一标识
#   - Qwen2.5-VL 是阿里开源的视觉语言模型，支持图文对话
#   - 3B 参数版本约 6GB(GGUF/量化) ~ 7GB(FP16)

CACHE_DIR = "./model_cache"  # 下载后的模型存放目录（相对当前工作目录）

# 文本生成解码参数（控制输出质量与多样性）
GENERATE_KWARGS = {
    "max_new_tokens": 128,    # 最多新生成 512 个 token（防止无限生成）
    "do_sample": True,        # True=采样（有随机性）；False=贪心解码（确定性强）
    "temperature": 0.7,       # 温度：越低越确定、越高越多样；0.7 是常用折中
    "top_p": 0.9,             # 核采样：只从累积概率 90% 的 token 中采样
}


In [16]:
#阿里魔塔社区下载
from modelscope import snapshot_download

# 使用 ModelScope 对应的模型ID下载
def download_model() -> str:
    local_path = snapshot_download('Qwen/Qwen2.5-VL-3B-Instruct', local_dir=r'C:\all_models\Qwen--Qwen2.5-VL-3B-Instruct')

    print(f"模型已成功下载到本地路径: {local_path}")
    return local_path

In [17]:
# ------------------------------------------------------------------------------
# 4. 初始化：加载 processor 与 model（整个脚本只做一次，耗时较长）
# ------------------------------------------------------------------------------
def load_model(local_path: str):
    """
    根据本地路径加载 processor 与 model。
    返回 (processor, model, device)，供后续三种调用方式复用。

    参数:
        local_path: str - download_model() 返回的本地仓库路径

    返回:
        processor: AutoProcessor - 负责文本分词 + 图像预处理
        model    : AutoModelForVision2Seq - 多模态模型本体
        device   : str - 实际运行设备 ("cuda:0" 或 "cpu")
    """
    # 自动选择设备：有 NVIDIA GPU 用 cuda，否则退回 cpu
    #   - "cuda:0" 表示第 0 号 GPU；CPU 下量化/精度参数会被自动忽略
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    print(f"[初始化] 使用设备: {device}")

    # Processor = tokenizer(文本) + image_processor(图像) 的组合
    #   它封装了 chat_template 渲染、tokenizer.encode、图像 resize/归一化
    processor = AutoProcessor.from_pretrained(local_path, trust_remote_code=True)
    #   trust_remote_code=True : 允许执行模型仓库自带的 modeling_*.py（Qwen 需要）

    # 加载模型权重（safetensors 分片文件）
    #   torch_dtype="auto" : 自动沿用模型原始精度（通常 bfloat16/float16）
    #   device_map="auto"  : 自动把不同层分配到可用 GPU/CPU（需安装 accelerate）
    model = AutoModelForImageTextToText.from_pretrained(
        local_path,
        torch_dtype="auto",   # 自动精度：GPU 上常用 bfloat16 节省显存
        device_map="auto",    # auto 自动设备映射：单 GPU 时等价于放到 cuda:0
        trust_remote_code=True,
    )
    model.eval()  # 切换到推理模式（关闭 dropout 等训练专用层）
    print("[初始化] 模型与处理器加载完成\n")
    return processor, model, device


In [ ]:
# ------------------------------------------------------------------------------
# 5. 公共工具：把"文本/图片"构造成模型要求的 messages 并渲染成 prompt
# ------------------------------------------------------------------------------
def build_prompt(processor, utext: str, image_path: str | None = None) -> dict:
    """
    构造符合 chat_template 的输入，并渲染成模型可接受的格式。
    这是"字符→token"流程的第②③步（messages → chat_template → tokenize）。

    参数:
        processor : AutoProcessor - 用于 apply_chat_template
        utext      : str           - 用户输入的文本（问题）
        image_path: str | None    - 本地图片路径；为 None 表示纯文本输入

    返回:
        dict - 已 tokenize 并预处理好的模型输入（含 input_ids、pixel_values 等），
               可直接传入 model.generate()
    """
    # 5-1. 组装 messages（对话结构化，对应流程第②步）
    #   每条消息含 role(角色) 与 content(内容)；content 为列表，支持混合文本与图像
    if image_path is not None:
        # 多模态：content 里同时放 image(本地路径字符串) 与 utext
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image_path},  # 图像占位（processor 会读取并预处理）
                    {"type": "text", "text": utext},
                ],
            }
        ]
    else:
        # 纯文本：content 只有 text
        messages = [{"role": "user", "content": [{"type": "text", "text": utext}]}]

    # 5-2. 应用 chat_template 渲染成带特殊 token 的文本
    #   add_generation_prompt=True : 在末尾追加 "assistant" 起始标记，告诉模型"该你说了"
    text_prompt = processor.apply_chat_template(
        messages,
        tokenize=False,          # 先不切 token，返回纯字符串
        add_generation_prompt=True,
    )

    # 5-3. 加载图像（若有多模态）；纯文本时为 None
    #   Qwen2.5-VL 的 processor 对图像参数名为 "images"
    image = Image.open(image_path).convert("RGB") if image_path else None

    # 5-4. 统一 tokenize + 图像预处理
    #   return_tensors="pt" : 返回 PyTorch 张量
    inputs = processor(
        text=[text_prompt],   # 渲染后的 prompt 文本（列表，支持 batch）
        images=image,         # 图像对象；纯文本传 None
        return_tensors="pt",  # 返回张量格式："pt"=PyTorch
        # 加这两行，限制图片分辨率，防止 CPU 内存爆炸。强制把图片的 token 数（不是像素数）限制在这个范围内
        min_pixels=256 * 28 * 28,   # 约 200×200
        max_pixels=512 * 28 * 28,   # 约 280×280
    )
    return inputs


def move_to_device(inputs: dict, device: str) -> dict:
    """把预处理后的所有张量搬到目标设备（GPU/CPU），避免设备不一致报错。"""
    return {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}


In [19]:
# 下载模型
local_path = download_model()

2026-09-11 14:56:28,438 | INFO    | modelscope_hub.download | Downloading 15 files from Qwen/Qwen2.5-VL-3B-Instruct@master


Downloading:   0%|          | 0/15 [00:00<?, ?file/s]

模型已成功下载到本地路径: C:\all_models\Qwen--Qwen2.5-VL-3B-Instruct


In [20]:
# 加载模型
processor, model, device = load_model(local_path)

[初始化] 使用设备: cpu


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.


[初始化] 模型与处理器加载完成



==============================================================================
方式一：纯文本输入 + 非流式输出

纯文本 → 模型 → 一次性返回完整字符串（非流式）。
完整演示：build_prompt → generate → decode 三段式。

==============================================================================

In [21]:
# ① 构造输入（messages → chat_template → tokenize）
text="用一句话介绍一下大语言模型的推理流程。"
text_inputs = build_prompt(processor, text=text, image_path=None)
pprint.pprint(text_inputs)

{'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1]]),
 'input_ids': tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198,  11622, 105321, 109432,  26288,
         102064, 104949,   9370, 113272, 102054,   1773, 151645,    198, 151644,
          77091,    198]]),
 'mm_token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0]])}


In [22]:
text_inputs = move_to_device(text_inputs, device)
pprint.pprint(text_inputs)

{'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1]]),
 'input_ids': tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198,  11622, 105321, 109432,  26288,
         102064, 104949,   9370, 113272, 102054,   1773, 151645,    198, 151644,
          77091,    198]]),
 'mm_token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0]])}


In [23]:

print("=" * 60)
print("[方式一] 纯文本 · 非流式输出")
print("=" * 60)

# 模型生成（前向传播 → logits → 采样 → token_id 序列）
# generate() 内部完成自回归循环，直到遇到 stop token 或达到 max_new_tokens
with torch.no_grad():  # 关闭梯度计算，节省显存与计算
    text_generated_ids = model.generate(**text_inputs, **GENERATE_KWARGS)

# ③ 截断输入部分，只保留"新生成的"token（input_ids 在 generated 的前缀里）
#   input_ids.shape[1] 是原始 prompt 长度，从它之后切片即模型回答
new_text_token_ids = text_generated_ids[0, text_inputs["input_ids"].shape[1]:]

# ④ 反分词（流程第⑧步：token_id → 字符串）
text_output = processor.tokenizer.decode(new_text_token_ids, skip_special_tokens=True)
#   skip_special_tokens=True : 丢弃 <|im_end|> 等特殊 token，只留可读文本

print(f"[用户]:{text}")
print(f"[模型]: {text_output}\n")


[方式一] 纯文本 · 非流式输出
[用户]:用一句话介绍一下大语言模型的推理流程。
[模型]: 大语言模型的推理流程包括输入文本，通过预训练模型进行编码，然后使用解码器生成输出文本。



In [11]:
pprint.pprint(text_generated_ids)
print("\n")

text_output = processor.tokenizer.decode(text_generated_ids)
pprint.pprint(text_output)
print("\n")

text_output = processor.tokenizer.decode(text_generated_ids, skip_special_tokens=True)
pprint.pprint(text_output)
print("\n")

new_text_generated_ids = text_generated_ids[0, text_inputs["input_ids"].shape[1]:]
# 等价
#new_text_token_ids = text_generated_ids[0][text_inputs["input_ids"].shape[1]:]
pprint.pprint(new_text_token_ids)
print("\n")
text_output = processor.tokenizer.decode(new_text_generated_ids, skip_special_tokens=True)

pprint.pprint(text_output)

tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198,  11622, 105321, 109432,  26288,
         102064, 104949,   9370, 113272, 102054,   1773, 151645,    198, 151644,
          77091,    198,  26288, 102064, 104949,   9370, 113272, 102054, 100630,
          31196, 108704,   5373,  67338,  98841, 104034, 104949,  43959,  72881,
          64559, 105790,   9370,  69041,  32757,  51463,   3837, 101889,  37029,
          98841, 104034, 104949,  71817, 104538,  57191,  43959,  66017,   1773,
         151645]], device='cuda:0')


['<|im_start|>system\n'
 'You are a helpful assistant.<|im_end|>\n'
 '<|im_start|>user\n'
 '用一句话介绍一下大语言模型的推理流程。<|im_end|>\n'
 '<|im_start|>assistant\n'
 '大语言模型的推理流程包括输入文本、通过预训练模型生成语义相似的向量表示，然后使用预训练模型进行预测或生成输出。<|im_end|>']


['system\n'
 'You are a helpful assistant.\n'
 'user\n'
 '用一句话介绍一下大语言模型的推理流程。\n'
 'assistant\n'
 '大语言模型的推理流程包括输入文本、通过预训练模型生成语义相似的向量表示，然后使用预训练模型进行预测或生成输出。']


tensor([ 2628

==============================================================================
方式二：纯文本输入 + 流式输出（逐块打印，类似 ChatGPT 打字效果）
==============================================================================
"""
纯文本 → 模型 → 逐 token(块) 流式输出。
原理：用 TextIteratorStreamer 在独立线程里接收 generate 产出的 token，
     主线程逐个 decode 并 print，实现"边生成边显示"。
"""

In [12]:
# ① 构造输入（messages → chat_template → tokenize）
poesy="写一首10句的七言诗，主题是春天。"
poesy_inputs = build_prompt(processor, text=poesy, image_path=None)
pprint.pprint(poesy_inputs)

{'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1]]),
 'input_ids': tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198,  61443, 108462,     16,     15,
          99700,   9370,  99612,  77144, 100045,   3837, 100220,  20412, 105303,
           1773, 151645,    198, 151644,  77091,    198]]),
 'mm_token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0]])}


In [13]:
poesy_inputs = move_to_device(poesy_inputs, device)
pprint.pprint(poesy_inputs)

{'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0'),
 'input_ids': tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198,  61443, 108462,     16,     15,
          99700,   9370,  99612,  77144, 100045,   3837, 100220,  20412, 105303,
           1773, 151645,    198, 151644,  77091,    198]], device='cuda:0'),
 'mm_token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0]], device='cuda:0')}


In [14]:

print("=" * 60)
print("[方式二] 纯文本 · 流式输出")
print("=" * 60)

from transformers import TextIteratorStreamer
from threading import Thread

# ② 创建流式器：负责把模型新生成的 token 缓存成可迭代的文本流
#   skip_prompt=True : 跳过 prompt 部分，只流式输出回答
#   skip_special_tokens=True : 输出时丢弃特殊 token
streamer = TextIteratorStreamer(
    processor.tokenizer,
    skip_prompt=True,
    skip_special_tokens=True,
)

# ③ 在子线程里运行 generate（生成过程会不断把 token 推给 streamer）
generation_kwargs = dict(**poesy_inputs, **GENERATE_KWARGS, streamer=streamer)
thread = Thread(target=model.generate, kwargs=generation_kwargs)
thread.start()

# ④ 主线程：从 streamer 逐块读取并打印（流程第⑧⑨步的流式版本）
print(f"[用户]: \n {poesy} \n[模型]:", flush=True)
for chunk in streamer:        # chunk: str，每次若干字符/token
    print(chunk, end="", flush=True)  # 立即刷到终端，形成打字机效果
thread.join()  # 等待生成线程结束
print("\n")


[方式二] 纯文本 · 流式输出
[用户]: 
 写一首10句的七言诗，主题是春天。 
[模型]:
春风拂面柳丝长，桃花笑映绿水中。
燕子归来筑巢忙，蜜蜂采蜜花间舞。
碧空如洗云无踪，山川秀美鸟语浓。
溪水潺潺润物生，春色满园人兴隆。



==============================================================================
方式三：多模态输入（本地图片）+ 文本输出（图片描述）
==============================================================================

"""
本地图片 + 文本问题 → 多模态模型 → 图片描述（字符串）。
流程与方式一相同，仅 build_prompt 时传入 image_path。
"""

In [15]:
# 下载图片或者读本地图片
import urllib.request
image_path = "./demo_image.jpg"
urllib.request.urlretrieve("https://picsum.photos/id/2/200", image_path)  # 下载一张测试图

('./demo_image.jpg', <http.client.HTTPMessage at 0x7f078abbecc0>)

In [16]:
# ① 构造多模态输入（content 含 image + text）
question="请详细描述这张图片的内容。"
question_inputs = build_prompt(processor, text=question, image_path=image_path)
pprint.pprint(question_inputs)

{'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]),
 'image_grid_thw':

In [17]:
question_inputs = move_to_device(question_inputs, device)
pprint.pprint(question_inputs)

{'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='cuda

In [18]:

print("=" * 60)
print("[方式三] 多模态 · 图片输入 → 文本输出")
print("=" * 60)

# ② 模型生成
with torch.no_grad():
    question_generated_ids = model.generate(**question_inputs, **GENERATE_KWARGS)

# ③ 截断 + 解码
new_question_generated_ids = question_generated_ids[0, question_inputs["input_ids"].shape[1]:]
output_text = processor.tokenizer.decode(new_question_generated_ids, skip_special_tokens=True)

print(f"\n[图片]: {image_path}")
print(f"[问题]: {question}")
print(f"[模型]: {output_text}\n")


[方式三] 多模态 · 图片输入 → 文本输出

[图片]: ./demo_image.jpg
[问题]: 请详细描述这张图片的内容。
[模型]: 这张图片展示了一张桌子的局部视角，桌面上铺着一张棕色的桌布。桌子上有一个白色的笔记本电脑，屏幕关闭着，旁边是一本黑色封面的笔记本和一支黑色笔。笔记本电脑右侧还有一个黑色的手机。在手机旁边的白色盘子里放着一杯咖啡，咖啡杯里有一把勺子，里面还有几块饼干。



In [22]:
streamer = TextIteratorStreamer(
    processor.tokenizer,
    skip_prompt=True,
    skip_special_tokens=True,
)

generation_kwargs = dict(**question_inputs, **GENERATE_KWARGS, streamer=streamer)
thread = Thread(target=model.generate, kwargs=generation_kwargs)
thread.start()

print(f"[问题]: {question}\n[模型]: ", end="", flush=True)
for chunk in streamer:
    print(chunk, end="", flush=True)
thread.join()
print("\n")

[问题]: 请详细描述这张图片的内容。
[模型]: 这张图片展示了一个办公或学习的场景。画面中央是一台笔记本电脑，屏幕朝上打开。键盘和触控板清晰可见，显示着黑色的键帽和触控板区域。

在笔记本电脑的右侧，有一部手机放在木质桌子上，手机是黑色的。手机旁边有一个小圆盘，可能是咖啡杯的一部分。杯子内似乎装有一些棕色的液体，可能是一杯咖啡或茶。

在笔记本电脑下方的木质桌子上，可以看到两个白色的杯子，杯子之间夹着一个勺子，看起来像是用于搅拌饮品用的。杯子的边缘有细密的装饰线条，增添

